In [19]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [20]:
# Chat Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=None, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    if temperature:
        params["temperature"] = temperature

    message = client.messages.create(**params)
    return message.content[0].text

In [ ]:
import re
import json
import ast

def validate_python(code):
    try:
        ast.parse(code)
        return 10
    except Exception as e:
        return 0

def validate_json(code):
    try:
        json.loads(code)
        return 10
    except Exception as e:
        return 0


def validate_regex(code):
    try:
        re.compile(code)
        return 10
    except Exception as e:
        return 0       
        

def grade_syntax(test_case, output):
    format = test_case["format"]
    if format == "python":
        return validate_python(output)
    elif format == "json":
        return validate_json(output)
    elif format == "regex":
        return validate_regex(output)

In [ ]:
# Function to grade a test case + output using a model
import json

def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaluate the solution:
<solution_criteria>
{test_case["solution_criteria"]}
</solution_criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)


In [ ]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
        Please solve the following task:

        {test_case["task"]}

        * Respond only with Python code, JSON, or a plain Regex.
        * Do not include any other text or comments or explanations.
        """
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```")
    output = chat(messages, stop_sequences=["```code"])
    return output


In [ ]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    model_grade = grade_by_model(test_case, output)
    syntax_grade = grade_syntax(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    syntax_score = syntax_grade["score"]

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [28]:
from statistics import mean
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each test case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean(result["score"] for result in results)
    print(f"Average score: {average_score}")

    return results

In [33]:
import json

with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 6.333333333333333


In [34]:
print(json.dumps(results, indent=4))

[
    {
        "output": "# AWS CloudWatch Log Parser\n\nHere's a comprehensive solution to parse AWS CloudWatch log entries:\n\n```python\nimport re\nfrom datetime import datetime\nfrom typing import Dict, Optional, Tuple\n\nclass CloudWatchLogParser:\n    \"\"\"Parse AWS CloudWatch log entries using regex patterns\"\"\"\n    \n    # Common CloudWatch log format patterns\n    PATTERNS = {\n        'standard': r'(\\d{4}-\\d{2}-\\d{2}T\\d{2}:\\d{2}:\\d{2}\\.\\d+Z?)\\s+\\[(\\w+)\\]\\s+(.*)',\n        'json': r'\\{\"timestamp\":\"([^\"]+)\",\"level\":\"([^\"]+)\",\"message\":\"([^\"]+)\"\\}',\n        'custom': r'(\\d{4}-\\d{2}-\\d{2}\\s\\d{2}:\\d{2}:\\d{2})\\s-\\s(\\w+)\\s-\\s(.*)',\n        'lambda': r'(\\d{4}-\\d{2}-\\d{2}T\\d{2}:\\d{2}:\\d{2}\\.\\d+Z)\\s(\\w+)\\s(.*)',\n    }\n    \n    def __init__(self, pattern_type: str = 'standard'):\n        \"\"\"\n        Initialize parser with pattern type\n        \n        Args:\n            pattern_type: 'standard', 'json', 'custom', or 'l